# Новая работа

In [1]:
import os 
import warnings 
warnings.filterwarnings("ignore")
import numpy as np 
import seaborn as sns 
import joblib
import pandas as pd 
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# загрузка данных и feture engenieering

In [2]:
df = pd.read_csv("real_estate_data.csv")
print("размер данных ",df.shape)
print("========")
print("кол-во столбцов",df.columns.tolist())
print("========")
df.head(10)

размер данных  (403487, 17)
кол-во столбцов ['id', 'type', 'sub_type', 'start_date', 'end_date', 'listing_type', 'tom', 'building_age', 'total_floor_count', 'floor_no', 'room_count', 'size', 'address', 'furnished', 'heating_type', 'price', 'price_currency']


,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,room_count,size,address,furnished,heating_type,price,price_currency
0,1,Konut,Rezidans,12/10/18,1/9/19,2,30,0,20 ve üzeri,2,2+1,90.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,3500.0,TRY
1,2,Konut,Daire,2/13/19,NaN,1,14,0,20 ve üzeri,20 ve üzeri,1+0,43.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,490000.0,TRY
2,3,Konut,Daire,10/9/18,11/8/18,1,30,0,1,Yüksek Giriş,2+1,NaN,Tekirdağ/Çorlu/Reşadiye,NaN,Fancoil,155000.0,TRY
3,4,Konut,Rezidans,9/10/18,10/10/18,1,30,3,20 ve üzeri,20 ve üzeri,6+1,450.0,İstanbul/Beşiktaş/Levent,NaN,Fancoil,32500000.0,TRY
4,5,Konut,Rezidans,12/10/18,1/9/19,1,30,0,20 ve üzeri,2,2+1,90.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,1450000.0,TRY
5,6,Konut,Rezidans,11/9/18,12/9/18,1,30,2,10-20 arası,10,1+1,45.0,İstanbul/Maltepe/Altayçeşme,NaN,Fancoil,780000.0,TRY
6,7,Konut,Daire,1/4/19,NaN,2,54,0,20 ve üzeri,14,3+1,160.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,3750.0,TRY
7,8,Konut,Villa,10/3/18,1/3/19,1,92,0,4,NaN,4+1,NaN,İzmir/Urla/M. Fevzi Çakmak,NaN,Fancoil,1500000.0,TRY
8,9,Konut,Daire,2/16/19,NaN,1,11,NaN,2,Kot 2,3+1,140.0,Çanakkale/Ayvacık/Küçükkuyu Bld. (Mıhlı),NaN,Fancoil,1500000.0,TRY
9,10,Konut,Daire,12/26/18,12/26/18,1,0,1,1,Asma Kat,2+2,550.0,İstanbul/Fatih/Sarıdemir,NaN,Fancoil,84256.0,GBP


In [3]:
def parse_room(x):
    if pd.isna(x):
        return np.nan, np.nan
    x = str(x).replace(" ", "")
    if "+" in x:
        p = x.split("+")
        try:
            return float(p[0]) + float(p[1]), float(p[1])
        except ValueError:
            return np.nan, np.nan
    try:
        return float(x), np.nan
    except ValueError:
        return np.nan, np.nan


def parse_age(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().replace(" ", "")
    is_range = "arası" in x or "arasi" in x
    nums, cur = [], ""
    for ch in x:
        if ch.isdigit():
            cur += ch
        else:
            if cur:
                nums.append(float(cur))
                cur = ""
    if cur:
        nums.append(float(cur))
    if not nums:
        return np.nan
    if is_range and len(nums) >= 2:
        return (nums[0] + nums[1]) / 2
    return nums[0]


def feature_engineer(data):
    d = data.copy()
    if "room_count" in d.columns:
        parsed = d["room_count"].apply(parse_room)
        d["total_rooms"]  = [x[0] for x in parsed]
        d["living_rooms"] = [x[1] for x in parsed]
    if "building_age" in d.columns:
        d["building_age_num"] = d["building_age"].apply(parse_age)

    for c in ["start_date", "end_date"]:
        if c in d.columns:
            d[c] = pd.to_datetime(d[c], errors="coerce")

    if "start_date" in d.columns and "end_date" in d.columns:
        d["listing_days"] = (d["end_date"] - d["start_date"]).dt.days
    if "start_date" in d.columns:
        d["start_year"]  = d["start_date"].dt.year
        d["start_month"] = d["start_date"].dt.month
        d["start_dow"]   = d["start_date"].dt.dayofweek

    drop = [c for c in ["start_date", "end_date", "room_count",
                        "building_age", "furnished", "id"]
            if c in d.columns]
    return d.drop(columns=drop, errors="ignore")


model_df = feature_engineer(df)

TARGET = "price"
model_df = model_df[model_df[TARGET].notna() & (model_df[TARGET] > 0)].copy()
print("После очистки таргета:", model_df.shape)

После очистки таргета: (402727, 18)


здесь мы меняем так чтобы время недвижемости в рынке и кол-во комнат были ввиде чисел 

In [4]:
X = model_df.drop(columns=[TARGET])
Y = model_df[TARGET]

num_features = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_features = [c for c in X.columns if c not in num_features]

print(num_features)
print("==========")
print(cat_features)

['listing_type', 'tom', 'size', 'total_rooms', 'living_rooms', 'building_age_num', 'listing_days', 'start_year', 'start_month', 'start_dow']
['type', 'sub_type', 'total_floor_count', 'floor_no', 'address', 'heating_type', 'price_currency']


# Проверка и пропрацессинг

In [5]:
random_state = 42

X_train, X_temp, y_train, y_temp = train_test_split(
    X, Y, test_size=0.30, random_state=random_state
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=random_state
)
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)

TOP_N, THRESHOLD = 50, 200
top_per_col = {}

for c in cat_features:
    n = X_train[c].nunique()
    if n > THRESHOLD:
        top = X_train[c].value_counts().head(TOP_N).index
        top_per_col[c] = top
        X_train[c] = X_train[c].where(X_train[c].isin(top), "other")
        X_val[c]   = X_val[c].where(X_val[c].isin(top), "other")
        X_test[c]  = X_test[c].where(X_test[c].isin(top), "other")
        print(f"  {c}: {n} → {X_train[c].nunique()}")
    else:
        print(f"  {c}: {n} — оставлено")

Train: (281908, 17) | Val: (60409, 17) | Test: (60410, 17)
  type: 1 — оставлено
  sub_type: 12 — оставлено
  total_floor_count: 12 — оставлено
  floor_no: 44 — оставлено
  address: 7278 → 51
  heating_type: 16 — оставлено
  price_currency: 4 — оставлено


In [6]:
LOW, HIGH = 50_000, 20_000_000

mask_train = (y_train >= LOW) & (y_train <= HIGH)
mask_val   = (y_val   >= LOW) & (y_val   <= HIGH)
mask_test  = (y_test  >= LOW) & (y_test  <= HIGH)

X_train, y_train = X_train[mask_train], y_train[mask_train]
X_val,   y_val   = X_val[mask_val],     y_val[mask_val]
X_test,  y_test  = X_test[mask_test],   y_test[mask_test]

print("Train:", len(y_train), "| Val:", len(y_val), "| Test:", len(y_test))
print("y_train max:", y_train.max(), "| min:", y_train.min())

Train: 199679 | Val: 42764 | Test: 42840
y_train max: 20000000.0 | min: 50000.0


In [7]:
for c in cat_features:
    X_train[c] = X_train[c].astype(str)
    X_val[c]   = X_val[c].astype(str)
    X_test[c]  = X_test[c].astype(str)

for c in num_features:
    X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
    X_val[c]   = pd.to_numeric(X_val[c],   errors="coerce")
    X_test[c]  = pd.to_numeric(X_test[c],  errors="coerce")

num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh",  OneHotEncoder(handle_unknown="ignore",
                          min_frequency=5,
                          sparse_output=True)),
])
pre = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features),
])

X_train_p = pre.fit_transform(X_train)
X_val_p   = pre.transform(X_val)
X_test_p  = pre.transform(X_test)
print("После препроцессинга:", X_train_p.shape)

После препроцессинга: (199679, 141)


In [8]:
def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    m = y_true != 0
    return np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100

def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    m = denom != 0
    return np.mean(2 * np.abs(y_pred[m] - y_true[m]) / denom[m]) * 100

def make_metrics(name, y_true, y_pred):
    return {
        "Модель":  name,
        "MAE":     mean_absolute_error(y_true, y_pred),
        "RMSE":    np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE %":  mape(y_true, y_pred),
        "sMAPE %": smape(y_true, y_pred),
        "R2":      r2_score(y_true, y_pred),
    }

In [9]:
# %%
y_train_log = np.log1p(y_train)
models = {}

ridge = Ridge(alpha=1.0, random_state=random_state)
ridge.fit(X_train_p, y_train_log)
models["Ridge"] = ridge

rf = RandomForestRegressor(n_estimators=100, max_depth=12,
                           min_samples_leaf=10, max_features="sqrt",
                           n_jobs=1, random_state=random_state)
rf.fit(X_train_p, y_train_log)
models["RandomForest"] = rf

print("Обучены:", list(models.keys()))

Обучены: ['Ridge', 'RandomForest']


In [11]:
val_rows = []
for name, m in models.items():
    p = np.expm1(m.predict(X_val_p))
    val_rows.append(make_metrics(name, y_val, p))

val_df = pd.DataFrame(val_rows).set_index("Модель").round(3)
print(val_df.to_string())

                     MAE        RMSE  MAPE %  sMAPE %     R2
Модель                                                      
Ridge         207972.567  738585.480  40.826   35.998  0.240
RandomForest  213839.258  778570.096  42.089   37.752  0.156


здесь можем видеть что у ridge как всегда получились лучше r2 и общий процент т абсалютное кол-во ошибок меньше всего

In [12]:
test_rows = []
for name, m in models.items():
    p = np.expm1(m.predict(X_test_p))
    test_rows.append(make_metrics(name, y_test, p))

test_metrics = pd.DataFrame(test_rows).set_index("Модель").round(3)
print(test_metrics.to_string())

final_name  = test_metrics["RMSE"].idxmin()
final_model = models[final_name]
print(f"\nФинальная модель: {final_name}")
print(test_metrics.loc[final_name])

                     MAE        RMSE  MAPE %  sMAPE %     R2
Модель                                                      
Ridge         204787.556  730408.245  40.100   35.844  0.220
RandomForest  209379.806  763887.222  41.582   37.608  0.146

Финальная модель: Ridge
MAE        204787.556
RMSE       730408.245
MAPE %         40.100
sMAPE %        35.844
R2              0.220
Name: Ridge, dtype: float64


Видим что ridge получился лучше всех и совершил меньше ошибок

In [13]:
os.makedirs("data", exist_ok=True)
joblib.dump(final_model,  "data/final_model.pkl")
joblib.dump(pre,          "data/preprocessor.pkl")
joblib.dump(num_features, "data/numeric_features.pkl")
joblib.dump(cat_features, "data/categorical_features.pkl")
joblib.dump(top_per_col,  "data/top_per_col.pkl")
test_metrics.to_csv("data/model_comparison_test.csv")
print("Сохранено в data/")

Сохранено в data/


In [17]:
TEST_PATH = "test_real_estate_samples.csv"
OUT_PATH  = "result_real_estate_samples.csv"

test_raw = pd.read_csv(TEST_PATH)
print("Размер:", test_raw.shape)
print("Столбцы:", test_raw.columns.tolist())
display(test_raw)

# Фильтр Satılık
test_sale = test_raw[test_raw["listing_type"] == 1].copy()
print(f"Satılık: {len(test_raw)} → {len(test_sale)}")
work = feature_engineer(test_sale)

Размер: (20, 17)
Столбцы: ['id', 'type', 'sub_type', 'start_date', 'end_date', 'listing_type', 'tom', 'building_age', 'total_floor_count', 'floor_no', 'room_count', 'size', 'address', 'furnished', 'heating_type', 'price', 'price_currency']


,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,room_count,size,address,furnished,heating_type,price,price_currency
0,60624,Konut,Daire,11/20/18,2/23/19,1,95,6-10 arası,5,1,2+1,120.0,Antalya/Alanya/Oba,NaN,Klima,240000.0,TRY
1,401385,Konut,Müstakil Ev,11/27/18,2/1/19,1,66,NaN,NaN,NaN,5+1,350.0,Muğla/Bodrum/Gündoğan,NaN,NaN,3850000.0,TRY
2,207139,Konut,Daire,12/8/18,2/4/19,1,58,0,5,3,2+1,80.0,Manisa/Yunusemre/Muradiye,NaN,Kombi (Doğalgaz),120000.0,TRY
3,51586,Konut,Daire,1/11/19,NaN,1,47,0,5,Kot 1,1+1,45.0,İstanbul/Kağıthane/Emniyet Evleri,NaN,Klima,185000.0,TRY
4,168183,Konut,Daire,11/24/18,NaN,1,95,0,4,1,3+2,130.0,İstanbul/Sancaktepe/Yenidoğan,NaN,Kombi (Doğalgaz),418000.0,TRY
5,118389,Konut,Daire,9/29/18,10/29/18,1,30,6-10 arası,4,Kot 1,3+1,NaN,Ankara/Mamak/Kutlu,NaN,Kombi (Doğalgaz),138000.0,TRY
6,286003,Konut,Daire,10/18/18,12/27/18,1,70,0,5,2,2+1,NaN,İstanbul/Büyükçekmece/Pınartepe,NaN,Kombi (Doğalgaz),290000.0,TRY
7,35135,Konut,Villa,9/4/18,12/19/18,1,106,0,2,Müstakil,3+1,190.0,Muğla/Bodrum/Yalıkavak,NaN,Klima,2350000.0,TRY
8,107300,Konut,Daire,11/1/18,NaN,1,118,6-10 arası,5,Bahçe katı,2+1,NaN,İstanbul/Ümraniye/Yamanevler,NaN,Kombi (Doğalgaz),270000.0,TRY
9,299199,Konut,Daire,1/22/19,2/21/19,2,30,4,8,9,2+1,75.0,İstanbul/Pendik/Yenişehir,NaN,Merkezi Sistem (Isı Payı Ölçer),1200.0,TRY


Satılık: 20 → 15


In [ ]:
for c in num_features + cat_features:
    if c not in work.columns:
        work[c] = np.nan

for c, top in top_per_col.items():
    if c in work.columns:
        work[c] = work[c].astype(str)
        work[c] = work[c].where(work[c].isin(top.astype(str)), "other")

X_new = work[num_features + cat_features].copy()
for c in num_features:
    X_new[c] = pd.to_numeric(X_new[c], errors="coerce")
for c in cat_features:
    X_new[c] = X_new[c].astype(str)

X_new_p = pre.transform(X_new)
y_pred  = np.expm1(final_model.predict(X_new_p))

print("Прогнозы:", np.round(y_pred, 2))
print("Min:", f"{y_pred.min():,.2f}", "| Max:", f"{y_pred.max():,.2f}")

Прогнозы: [264131.04 620535.01 222332.6  154082.75 349505.21 205926.5  220109.53
 820415.42 218945.3  242762.23 871613.5  175173.45 397034.28 281705.34
 269198.17]
Min: 154,082.75 | Max: 871,613.50


In [ ]:
result_df = test_sale.copy()
result_df["Предсказанная цена"] = np.round(y_pred, 2)

TARGET_COL = None
for c in ["price", "fiyat", "Price", "Fiyat", "target"]:
    if c in test_sale.columns:
        TARGET_COL = c
        break

if TARGET_COL is not None:
    y_true = test_sale[TARGET_COL].values.astype(float)
    result_df["Фактическая цена"] = y_true

    abs_err = np.abs(y_true - y_pred)
    rel_err = abs_err / np.where(y_true == 0, np.nan, y_true) * 100
    result_df["Абс. ошибка"]    = np.round(abs_err, 2)
    result_df["Отн. ошибка, %"] = np.round(rel_err, 2)

    print(f"MAE:  {mean_absolute_error(y_true, y_pred):,.2f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):,.2f}")
    print(f"R²:   {r2_score(y_true, y_pred):.4f}")
    print(f"Средняя отн. ошибка: {np.nanmean(rel_err):.2f}%")
    print(f"Медианная отн. ошибка: {np.nanmedian(rel_err):.2f}%")
else:
    print("Ground-truth не найден")

result_df["ФИО"]    = "Имре Эмиль Бахаттинович"
result_df["Группа"] = "23п-4"
result_df.to_csv(OUT_PATH, index=False)
print(f"Сохранено: {OUT_PATH}")
display(result_df)

MAE:  387,318.99
RMSE: 926,688.25
R²:   0.1685
Средняя отн. ошибка: 45.78%
Медианная отн. ошибка: 39.31%
Сохранено: result_real_estate_samples.csv


,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,...,furnished,heating_type,price,price_currency,Предсказанная цена,Фактическая цена,Абс. ошибка,"Отн. ошибка, %",ФИО,Группа
0,60624,Konut,Daire,11/20/18,2/23/19,1,95,6-10 arası,5,1,...,NaN,Klima,240000.0,TRY,264131.04,240000.0,24131.04,10.05,Имре Эмиль Бахаттинович,23п-4
1,401385,Konut,Müstakil Ev,11/27/18,2/1/19,1,66,NaN,NaN,NaN,...,NaN,NaN,3850000.0,TRY,620535.01,3850000.0,3229464.99,83.88,Имре Эмиль Бахаттинович,23п-4
2,207139,Konut,Daire,12/8/18,2/4/19,1,58,0,5,3,...,NaN,Kombi (Doğalgaz),120000.0,TRY,222332.60,120000.0,102332.60,85.28,Имре Эмиль Бахаттинович,23п-4
3,51586,Konut,Daire,1/11/19,NaN,1,47,0,5,Kot 1,...,NaN,Klima,185000.0,TRY,154082.75,185000.0,30917.25,16.71,Имре Эмиль Бахаттинович,23п-4
4,168183,Konut,Daire,11/24/18,NaN,1,95,0,4,1,...,NaN,Kombi (Doğalgaz),418000.0,TRY,349505.21,418000.0,68494.79,16.39,Имре Эмиль Бахаттинович,23п-4
5,118389,Konut,Daire,9/29/18,10/29/18,1,30,6-10 arası,4,Kot 1,...,NaN,Kombi (Doğalgaz),138000.0,TRY,205926.50,138000.0,67926.50,49.22,Имре Эмиль Бахаттинович,23п-4
6,286003,Konut,Daire,10/18/18,12/27/18,1,70,0,5,2,...,NaN,Kombi (Doğalgaz),290000.0,TRY,220109.53,290000.0,69890.47,24.10,Имре Эмиль Бахаттинович,23п-4
7,35135,Konut,Villa,9/4/18,12/19/18,1,106,0,2,Müstakil,...,NaN,Klima,2350000.0,TRY,820415.42,2350000.0,1529584.58,65.09,Имре Эмиль Бахаттинович,23п-4
8,107300,Konut,Daire,11/1/18,NaN,1,118,6-10 arası,5,Bahçe katı,...,NaN,Kombi (Doğalgaz),270000.0,TRY,218945.30,270000.0,51054.70,18.91,Имре Эмиль Бахаттинович,23п-4
10,120316,Konut,Daire,1/7/19,NaN,1,51,11-15 arası,4,Kot 2,...,NaN,Kombi (Doğalgaz),99999.0,TRY,242762.23,99999.0,142763.23,142.76,Имре Эмиль Бахаттинович,23п-4


Теперь всё намного стабильнее чем в прошлый раз и мы видим что хоть ошибки и есть они не настолько разбросаны каак в прошлый раз